# Fundamentals

structure
1. How to connect with the models
2. how to connect with the data
3. data chaining

#### langchaing - chaing

**Store**
Database(data) -> Embedding model -> VectorialDB(data_embeddings)

**Query**
Store -> Prompt(human query) -> LLM(model) -> Response

#### Data processing

DB(data) -> Data Cleaning -> Abstract generation -> Metadata selection -> Text fragmentation -> Embedding model -> VectorialDB selection

# OpenSource Models


In [ ]:
%%capture
!pip install -q transformers accelerate einops

In [13]:
# pc specs
!echo "PC Specifications:"
!lscpu
!echo "RAM Specifications:"
!free -h
!echo "Graphics Specifications:"
!lspci | grep VGA

PC Specifications:
Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             39 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      8
  On-line CPU(s) list:       0-7
Vendor ID:                   GenuineIntel
  Model name:                11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz
    CPU family:              6
    Model:                   140
    Thread(s) per core:      2
    Core(s) per socket:      4
    Socket(s):               1
    Stepping:                1
    CPU(s) scaling MHz:      83%
    CPU max MHz:             4200.0000
    CPU min MHz:             400.0000
    BogoMIPS:                4838.40
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush dts acpi mmx fxsr sse 
                             sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm cons
                             tant_t

In [36]:
from transformers import AutoTokenizer, pipeline
import torch

# model = "tiiuae/falcon-7b-instruct"

model = "tiiuae/falcon-rw-1b"  # ~2GB, manageable on CPU/integrated graphics
tokenizer = AutoTokenizer.from_pretrained(model)

pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    dtype=torch.bfloat16,
    device_map="auto"
)

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 22503.62it/s]


In [37]:
type(pipeline)

transformers.pipelines.text_generation.TextGenerationPipeline

In [38]:
%%capture
!pip install -U langchain-huggingface

In [39]:
from langchain_huggingface import HuggingFacePipeline

llm_falcon = HuggingFacePipeline(
    pipeline=pipeline,
    model_kwargs={
        "temperature": 0.1,
        "max_length": 512,
        "top_p": 0.75, # top_p is the cumulative probability of parameter highest probability vocabulary tokens to keep for nucleus sampling
        "top_k": 40, # top_k is the number of highest probability vocabulary tokens to keep for top-k-filtering
        "do_sample": True,
        "num_return_sequences": 1,
        "eos_token_id": tokenizer.eos_token_id
    }
)

In [40]:
llm_falcon

HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14'}}, pipeline=TextGenerationPipeline: {'model': 'FalconForCausalLM', 'dtype': 'bfloat16', 'device': 'cpu', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='tiiuae/falcon-rw-1b', model_kwargs={'temperature': 0.1, 'max_length': 512, 'top_p': 0.75, 'top_k': 40, 'do_sample': True, 'num_return_sequences': 1, 'eos_token_id': 50256})

In [41]:
from langchain_core.prompts import PromptTemplate

template = """question: {question}
Response: We're going to think through this step by step."""
prompt = PromptTemplate.from_template(template)

chain = prompt | llm_falcon

response = chain.invoke({"question": "Why is the sky blue?"})
print(response)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


question: Why is the sky blue?
Response: We're going to think through this step by step. The first step will be to consider the following:
- Why is there light?
- Why does the sun shine?
- Why is the sky blue?
- Why does it rain?
- Why does the ocean swell and cause tides?
Before we answer these questions, let's start with a question: Why is there light? Light is made up of three main parts. The first is the energy of the source of light. The second part is the thing that we see. The third part is the motion of the light.
The energy of light comes from the sun. The sun is made up of hydrogen and helium. When the sun is shining, we feel warmer and lighter. This is because we are emitting a portion of our own light.
Let's think through the second part, the thing we see. The sun is made up of hydrogen and helium. The sun is a star. Stars are made up of hydrogen and helium. The atoms of hydrogen and helium are in a very small region of space. The atoms are very close together. This is beca

In [44]:
chain.invoke("What can you do?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'question: What can you do?\nResponse: We\'re going to think through this step by step.\n1. Identify your top three priorities.\n2. Identify what you can do to achieve those priorities.\n3. Take action on the choices you\'ve made.\n4. Measure your progress.\n5. Repeat.\n6. Measure your progress.\nIf you\'re like most people, you have a list of priorities. Some are higher than others. Some are more important to you than others. But if you find yourself asking, "What can I do to change my life?" here are some things to keep in mind:\n1. Don\'t change everything at once.\n2. Don\'t try to change everything at once.\n3. Don\'t try to change everything at once... yet.\n4. When you think you\'re done, think again.\n5. When you think you\'re done, think again.\n6. When you\'re done, think again.\n7. Keep at it: It\'s a marathon, not a sprint.\n8. Keep at it: It\'s a marathon, not a sprint.\n9. Keep at it: It\'s a marathon, not a sprint.\nIt\'s easy to get overwhelmed by what you can\'t do. Th